# 01: Veri temizliği

Bu not defterinde Letterboxd puanlarımı TMDB'den çektiğim film bilgileriyle birleştiriyorum. Sonraki iki not defteri (EDA ve model) tek bir tablodan çalışsın diye.

Girdiler: `data/raw/ratings.csv`, `data/raw/likes_films.csv`, `data/processed/films_enriched.csv`
Çıktı: `data/processed/analysis_table.csv`

> Bu ürün TMDB API'sini kullanır ancak TMDB tarafından onaylanmamıştır.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..').resolve()
RAW, PROC = ROOT / 'data' / 'raw', ROOT / 'data' / 'processed'

ratings = pd.read_csv(RAW / 'ratings.csv', parse_dates=['Date'])
likes = pd.read_csv(RAW / 'likes_films.csv')
films = pd.read_csv(PROC / 'films_enriched.csv')
print(ratings.shape, likes.shape, films.shape)
ratings.head()

(358, 5) (77, 4) (700, 16)


,Date,Name,Year,Letterboxd URI,Rating
0,2023-08-31,Knives Out,2019,https://boxd.it/jWEA,4.5
1,2022-12-05,Se7en,1995,https://boxd.it/29zs,5.0
2,2022-12-06,Inception,2010,https://boxd.it/1skk,4.0
3,2022-12-06,2001: A Space Odyssey,1968,https://boxd.it/2bf0,4.0
4,2023-07-04,Blade Runner 2049,2017,https://boxd.it/b8wK,4.0


## 1. Önce veriye bir bakalım

Birleştirmeden önce puan dosyasında yinelenen film, eksik değer ya da garip bir puan var mı diye bakıyorum.

In [2]:
print('Yinelenen film (ratings):', ratings['Letterboxd URI'].duplicated().sum())
print('Eksik değerler:\n', ratings.isna().sum())
print('Puan değerleri:', sorted(ratings['Rating'].unique()))

Yinelenen film (ratings): 0
Eksik değerler:
 Date              0
Name              0
Year              0
Letterboxd URI    0
Rating            0
dtype: int64
Puan değerleri: [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]


## 2. Puanlar ile TMDB'yi birleştirme

Birleşmeyen filmlerin çoğu dizi ya da bölüm çıkacak, TMDB'nin film aramasında bulunmuyorlar. Bu projede film zevkine bakıyorum, o yüzden onları analizden çıkarıyorum. Hangilerinin düştüğünü de yazdırıyorum ki gözden kaçmasın.

In [3]:
df = ratings.rename(columns={'Letterboxd URI': 'uri', 'Name': 'name', 'Year': 'year',
                             'Rating': 'rating', 'Date': 'rated_date'})
df = df.merge(films.drop(columns=['name', 'year']), on='uri', how='left', indicator=True)

dropped = df[df['_merge'] == 'left_only']
print(f'TMDB eşleşmeyen (analiz dışı): {len(dropped)}')
print(dropped['name'].tolist())

df = df[df['_merge'] == 'both'].drop(columns='_merge').reset_index(drop=True)
print('Analize giren film:', len(df))

TMDB eşleşmeyen (analiz dışı): 3
['11.22.63', "The Making of 'Dunkirk'", 'Black Mirror: Hang the DJ']
Analize giren film: 355


## 3. Öznitelikler

Modelde ve testlerde kullanacağım sütunları burada türetiyorum.

In [4]:
# beğendiğim filmler (Letterboxd'de kalp verdiklerim)
df['liked'] = df['uri'].isin(likes['Letterboxd URI']).astype(int)

# zaman: on yıl, puanladığım sırada filmin kaç yaşında olduğu, puanlama yılı
df['decade'] = (df['year'] // 10) * 10
df['film_age_at_rating'] = df['rated_date'].dt.year - df['year']
df['rated_year'] = df['rated_date'].dt.year

# dil ve ülke
df['is_english'] = (df['original_language'] == 'en').astype(int)
df['primary_country'] = df['countries'].fillna('').str.split('|').str[0].replace('', np.nan)
df['is_turkish'] = (df['original_language'] == 'tr').astype(int)

# yönetmen: birden fazla varsa ilkini alıyorum
df['director'] = df['directors'].fillna('').str.split('|').str[0].replace('', np.nan)

# türler: her tür için ayrı 0/1 sütunu (bir filmin birden fazla türü olabiliyor)
genre_dummies = df['genres'].fillna('').str.get_dummies(sep='|').add_prefix('g_')
df['n_genres'] = genre_dummies.sum(axis=1)
df = pd.concat([df, genre_dummies], axis=1)

# oy sayısı ve popülerlik çok çarpık dağılıyor, log alıyorum
df['log_votes'] = np.log1p(df['tmdb_vote_count'])
df['log_popularity'] = np.log1p(df['popularity'])

# sınıflandırma hedefi: 4 ve üstü verdiysem "sevdim"
df['loved'] = (df['rating'] >= 4).astype(int)
df.shape

(355, 49)

## 4. Kontrol

Eksik değerlere, süre sütununa ve tür sayılarına bakıyorum. Az örnekli türleri ve sıfır süreli filmleri burada yakalamak istiyorum.

In [5]:
print(df[['rating','year','runtime','tmdb_vote_average','tmdb_vote_count','n_genres']].describe().round(2))
print('\nEksik değerler:')
print(df[['runtime','primary_country','director','tmdb_vote_average']].isna().sum())
print('\nSıfır süre:', (df['runtime'] == 0).sum())
print('\nYıl farkı (Letterboxd yılı - TMDB yılı) yok; yıl Letterboxd kaynaklı.')
print('\nTür sayıları:')
print(genre_dummies.sum().sort_values(ascending=False))

       rating     year  runtime  tmdb_vote_average  tmdb_vote_count  n_genres
count  355.00   355.00   355.00             355.00           355.00    355.00
mean     3.66  2007.40   125.90               7.46         10119.50      2.48
std      0.91    14.71    32.16               0.69          8593.58      0.83
min      0.50  1927.00    23.00               4.31            18.00      1.00
25%      3.00  2000.00   105.00               7.03          3492.50      2.00
50%      4.00  2011.00   122.00               7.52          7883.00      3.00
75%      4.50  2018.00   139.00               7.99         15280.50      3.00
max      5.00  2026.00   432.00               8.74         41996.00      5.00

Eksik değerler:
runtime              0
primary_country      1
director             1
tmdb_vote_average    0
dtype: int64

Sıfır süre: 0

Yıl farkı (Letterboxd yılı - TMDB yılı) yok; yıl Letterboxd kaynaklı.

Tür sayıları:
g_Drama              206
g_Thriller            82
g_Science Fiction     77


## 5. Notlar

- Sadece bir filmini izlediğim yönetmen modelde bir şey anlatmaz. Modellemede en az 3 filmini izlediğim yönetmenleri ayrı kategori yapacağım, gerisi "diğer" olacak.
- Puanlar yarım yarım artıyor ve sıralı bir ölçek. Testlerde parametrik olmayan yöntemleri kullanacağım.
- Süresi 0 görünen film varsa onu eksik değer sayıyorum. Bu hücre onu NaN yapıp tabloyu kaydediyor.

In [6]:
df.loc[df['runtime'] == 0, 'runtime'] = np.nan
out = PROC / 'analysis_table.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')
print('Kaydedildi:', out, df.shape)

Kaydedildi: D:\claude\letterboxd-projesi\data\processed\analysis_table.csv (355, 49)
